# Phase 4 — GAT (mạng chú ý trên đồ thị bệnh nhân)

**Chạy Phase 3 trước notebook này.** Phase 4 cần file `phase3_oof_<nguồn>.npz` để đối
chiếu trực tiếp trên cùng bệnh nhân.

| Việc | Mục đích |
|---|---|
| GAT trên `control` | Kết quả chính |
| GAT tắt đồ thị (`--no-graph`) | Ablation: đồ thị đóng góp bao nhiêu |
| GAT trên `frozen` | Kết luận có phụ thuộc bộ đặc trưng ảnh không |

**GAT phải vượt được gì:** hơn B3/B3b ở nhóm **không có ảnh** (đó là lý do GNN tồn tại),
và không thua B2/L2 ở nhóm có ảnh. Notebook tự in bảng so sánh kèm khoảng tin cậy.

> Không cần cài `torch_geometric` — lớp GATv2 được cài bằng torch thuần trong
> `fusion/gnn.py`.

## 1. Bật GPU

**Runtime → Change runtime type → T4 GPU → Save**, rồi chạy cell dưới.

In [ ]:
import torch
print("Torch:", torch.__version__)
print("GPU  :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG THẤY GPU — vào Runtime → Change runtime type → T4 GPU")

## 2. Tải mã nguồn và dữ liệu

In [ ]:
import os, shutil
if os.path.exists("/content/repo"):
    shutil.rmtree("/content/repo")
!git clone -q -b fusion_gnn https://github.com/Do-Trung-Quan/DO-AN-TOT-NGHIEP-E22-T02.git /content/repo
os.chdir("/content/repo")

need = ["fusion/output/fusion_dataset_ketqua_control.npz",
        "fusion/output/graph_ketqua_k10_bridge.npz",
        "fusion/output/tabular_ketqua.npz",
        "output/ketqua/ts_predictions.parquet"]
missing = [f for f in need if not os.path.exists(f)]
print("THIẾU:", missing) if missing else print("✅ Đủ file đầu vào")

## 3. Chạy lại Phase 3 (bắt buộc)

Phase 3 đã được sửa 3 chỗ kể từ lần b chạy trước:

1. **Sửa lỗi hàng xóm một chiều** — trước đây mỗi node chỉ thấy 1,1 hàng xóm có ảnh
   thay vì 3,23. Số cũ là của phiên bản bị thiệt thòi.
2. **Thêm baseline hồi quy logistic** (L1, L2, L3b) — vì mô hình sâu đang thua mô hình
   tuyến tính, phải đưa vào bảng chứ không giấu.
3. **Thêm B3b** — mượn *điểm nguy cơ* của hàng xóm thay vì mượn cả vector 256 chiều.
4. **Cổng xét bằng khoảng tin cậy**, không so hai số thập phân trần.

Khoảng 10–15 phút cho cả hai nguồn.

In [ ]:
!python fusion/baselines.py --source control --seeds 3 --epochs 60
!python fusion/baselines.py --source frozen  --seeds 3 --epochs 60

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("fusion/output/phase3_baselines_control.md", encoding="utf-8").read()))

## 4. GAT trên nguồn `control` — kết quả chính

Khoảng 5–10 phút.

In [ ]:
!python fusion/gnn.py --source control --seeds 3 --epochs 300

In [ ]:
display(Markdown(open("fusion/output/phase4_gat_control.md", encoding="utf-8").read()))

## 5. Ablation — tắt truyền tin trên đồ thị

Cùng kiến trúc, cùng số tham số, chỉ khác là **không cho node nói chuyện với hàng xóm**.
Chênh lệch giữa cell 4 và cell này chính là đóng góp thuần của đồ thị.

In [ ]:
!python fusion/gnn.py --source control --seeds 3 --epochs 300 --no-graph

In [ ]:
display(Markdown(open("fusion/output/phase4_gat_nograph_control.md", encoding="utf-8").read()))

## 6. GAT trên nguồn `frozen`

Phase 3 cho thấy đồ thị chỉ giúp với `control`, không giúp với `frozen`. Cell này kiểm
tra xem GAT có khắc phục được điều đó không.

In [ ]:
!python fusion/gnn.py --source frozen --seeds 3 --epochs 300

In [ ]:
display(Markdown(open("fusion/output/phase4_gat_frozen.md", encoding="utf-8").read()))

## 7. Tải toàn bộ kết quả về máy

Copy vào `fusion/output/` ở repo local rồi commit. **Không `git push` từ Colab.**

In [ ]:
from google.colab import files
import glob
for f in sorted(glob.glob("fusion/output/phase3_*") + glob.glob("fusion/output/phase4_*")):
    print("Đang tải:", f)
    files.download(f)